# Example: BBBC014 — Dose-Response Morphological Profiling

Morphological profiling of distinct cell lines (A549 and MCF7) treated with various compounds across multiple concentrations to study dose-response effects.

**Dataset:** [BBBC014](https://bbbc.broadinstitute.org/BBBC014) (2-channel compound dose-response)

**Key Steps:**
1. **Metadata Prep:** Process annotations and calculate log concentrations for dose responses.
2. **Multi-Channel Processing:** Process full field-of-view 2-channel images.
3. **Dose-Response Analysis:** Extract embeddings and compute properties to differentiate cell line responses and visualize dose-dependent morphological shifts.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

Define the paths to the BBBC014 image dataset and the metadata CSV containing compound and concentration information.

In [ ]:
DATA_DIR = "/datasets/BBBC/BBBC014/BBBC014_v1_images"
METADATA_PATH = "/datasets/BBBC/BBBC014/metadata.csv"

## 2. Load Metadata & Preprocessing

We load the metadata and map the cell types to their respective line names (MCF7 and A549). We also calculate log concentrations to facilitate dose-response visualization.

In [ ]:
df_metadata = pd.read_csv(METADATA_PATH)

# Map cell types to line names
df_metadata["cell_line"] = df_metadata["cell_type"].apply(
    lambda x: "MCF7" if x in ["A", "B", "C", "D"] else "A549"
)

# Map concentration points to log concentrations
map_concentration = {
    1: -7.0, 2: -7.523, 3: -8.0, 4: -8.523, 5: -9.0, 6: -9.523, 
    7: -10.0, 8: -10.523, 9: -11.0, 10: -11.523, 11: -12.0, 12: -13.0
}
df_metadata["log_concentration"] = df_metadata["concentration_point"].map(map_concentration)

df_metadata.head()

## 3. Multi-Channel Data Discovery

Since BBBC014 uses two channels per image, we specify both filename columns. PhenoMe will automatically load and merge these channels for processing.

In [ ]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    str(DATA_DIR),
    metadata_fn=make_dataframe_metadata_fn(
        df_metadata, 
        filename_column=["filename_channel_1", "filename_channel_2"]
    ),
)
file_df.head()

## 4. Feature Extraction

We extract DINOv2 embeddings and compute a full set of morphological properties. Note the use of large `resize_size` and `pad_size` to accommodate full field-of-view images.

In [ ]:
pheno.process_images(
    wrapper,
    batch_size=8,
    channel_mode="split",
    resize_size=1358,
    pad_size=1360,
)

df_properties = pheno.compute_properties(
    property_preset="complete",
    n_jobs=4,
)

## 5. Visualization

Visualize the morphological differences between cell lines and observe dose-dependent shifts using t-SNE and UMAP.

In [ ]:
pheno.plot_tsne(
    color_by="cell_line",
    n_components=2,
)

In [ ]:
pheno.plot_umap(
    color_by="log_concentration",
    filters={"cell_line": "A549"},
    n_components=2,
)

## 6. Interactive Exploration

Launch the interactive explorer to inspect the multi-channel images and their relative positions in the phenospace, highlighting how treatments affect cell morphology.

In [ ]:
e = pheno.create_interactive_explorer()